# Vicuña · Experimentos T08 y T10

Misma red `Net` del starter. Cambia solo el entrenamiento: pérdida, número de épocas, promedio móvil de pesos (EMA), recorte de HR, ensamble de semillas y reentrenamiento con todo junio.

- **Etapa A:** pérdida × épocas, 2 semillas, validación temporal (16–30 jun). Época final fija: la validación no elige la época.
- **Etapa B:** ensamble de 5 semillas de la mejor configuración.
- **Etapa C:** reentrena con **todos** los datos (5 semillas) y escribe `submission.csv`. No se envía desde aquí.

Se compara con `score_oficial`. Línea base T02: 0.44092 (mejor época elegida en validación).

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
# COPIA EXACTA de train/validacion.py: no editar aquí
"""
Validación temporal con la métrica del concurso (T02 · HU-09). Solo NumPy.

Corte temporal fijo: validación = últimas dos semanas de junio (desde 2024-06-16 00 UTC);
entrenamiento = todo lo anterior. Nunca se hace split aleatorio.

Hay dos scores y no son comparables entre sí:
  score_local    el del starter: cada RMSE se divide entre el RMSE de la climatología de train
                 medido en la propia validación (climatología = 0). Todas las horas.
  score_oficial  fórmula del leaderboard, con sus constantes fijas, sobre las horas que Kaggle
                 evalúa (00, 03, ..., 21 UTC). Es el que se usa para comparar experimentos.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_validacion.py comprueba que ambas sean iguales.
"""
import numpy as np

VARS = ("RH", "T", "U", "V")
INICIO_VAL = "2024061600"
ESCALAS_OFICIALES = np.array([25.360, 2.0652, 2.6776, 2.0876])
HORAS_EVALUADAS = tuple(range(0, 24, 3))


def horas(times):
    """Hora UTC (0-23) de cada marca 'YYYYMMDDHH'."""
    return np.array([int(str(t)[-2:]) for t in times])


def corte_temporal(times, inicio_val=INICIO_VAL):
    """Máscaras booleanas (train, val): val son las marcas desde `inicio_val` en adelante."""
    val = np.asarray(times).astype(str) >= inicio_val
    trn = ~val
    if not trn.any() or not val.any():
        raise ValueError(f"El corte {inicio_val} deja vacío el entrenamiento o la validación.")
    return trn, val


def climatologia(Y, times):
    """Media por hora UTC: (24, filas, columnas, variables). Exige datos en las 24 horas."""
    h = horas(times)
    faltan = sorted(set(range(24)) - set(h.tolist()))
    if faltan:
        raise ValueError(f"No hay datos para las horas UTC {faltan}.")
    return np.stack([Y[h == k].mean(0) for k in range(24)])


def rmse_por_variable(pred, real):
    """RMSE de cada variable sobre todas las horas y celdas. Entradas (N, filas, columnas, 4)."""
    if pred.shape != real.shape:
        raise ValueError(f"Formas distintas: pred {pred.shape} y real {real.shape}.")
    rmse = np.sqrt(((pred - real) ** 2).mean(axis=(0, 1, 2)))
    if not np.isfinite(rmse).all():
        raise ValueError("Hay NaN o infinitos en la predicción o en el valor real.")
    return rmse


def score(rmse, escalas):
    """1 - media de rmse/escala; las cuatro variables pesan igual."""
    return float(1 - (rmse / escalas).mean())


def escala_climatologia(Y, times, inicio_val=INICIO_VAL):
    """RMSE en validación de la climatología calculada solo con train (denominador de score_local)."""
    t = np.asarray(times).astype(str)
    trn, val = corte_temporal(t, inicio_val)
    clim = climatologia(Y[trn], t[trn])
    return rmse_por_variable(clim[horas(t[val])], Y[val])


def validar(pred_val, Y, times, inicio_val=INICIO_VAL):
    """
    Evalúa una predicción de la validación.

    pred_val : (N_val, 70, 35, 4) en unidades físicas, en el orden de las horas de validación.
    Y, times : target y marcas de TODO train.npz (de ahí salen el corte y la climatología).
    """
    t = np.asarray(times).astype(str)
    _, val = corte_temporal(t, inicio_val)
    real, t_val = Y[val], t[val]
    rmse = rmse_por_variable(pred_val, real)
    evaluadas = np.isin(horas(t_val), HORAS_EVALUADAS)
    rmse_3h = rmse_por_variable(pred_val[evaluadas], real[evaluadas])
    return {
        "score_local": score(rmse, escala_climatologia(Y, t, inicio_val)),
        "rmse": {v: float(r) for v, r in zip(VARS, rmse)},
        "score_oficial": score(rmse_3h, ESCALAS_OFICIALES),
        "rmse_horas_evaluadas": {v: float(r) for v, r in zip(VARS, rmse_3h)},
        "horas_val": int(val.sum()),
        "horas_evaluadas": int(evaluadas.sum()),
        "primera": str(min(t_val)),
        "ultima": str(max(t_val)),
    }

In [ ]:
def windows(X, t):
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        prev = i
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out   # (n, 3, 17, 13, 4)

Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)
print(Wtr.shape, Wte.shape)

In [ ]:
import json, platform
assert torch.cuda.is_available(), 'Este notebook necesita GPU'
dev = 'cuda'
QUICK = False                      # True: prueba corta de que todo corre
trn, val = corte_temporal(ttr)
Wtr, Wte = (Wtr.astype(np.float32), Wte.astype(np.float32))
hour = horas
print(torch.cuda.get_device_name(0), '·', platform.python_version(), np.__version__, torch.__version__)
print(f'train {trn.sum()} h · val {val.sum()} h')

In [ ]:
# Coordenadas de la grilla WRF dentro de la grilla GFS, para grid_sample
glat, glon = st['gfs_lats'], st['gfs_lons']            # GFS: fila 0 = norte
wlat, wlon = st['wrf_lats'], st['wrf_lons']            # WRF: fila 0 = sur
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32, device=dev)   # (1,70,35,2)
DEM = torch.tensor((dem - dem.mean()) / dem.std(), dtype=torch.float32, device=dev)[None, None]

class Net(nn.Module):
    def __init__(self, w=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(
            nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,3,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = self.enc(torch.cat([x, hh], 1))
        f = F.grid_sample(f, GRID.expand(B, -1, -1, -1), align_corners=True)   # (B,w,70,35)
        f = torch.cat([f, DEM.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

In [ ]:
def to_t(a): return torch.tensor(a, dtype=torch.float32, device=dev)
ESC = to_t(ESCALAS_OFICIALES)

def datos_de(mascara):
    """Normalización y climatología SOLO con las horas de `mascara`; todo queda en la GPU."""
    mu_x = Xtr[mascara].mean((0, 1, 2)); sd_x = Xtr[mascara].std((0, 1, 2))
    clim = climatologia(Ytr[mascara], ttr[mascara])
    anom_sd = (Ytr[mascara] - clim[horas(ttr[mascara])]).std((0, 1, 2))
    idx = np.where(mascara)[0]
    h = to_t(horas(ttr[idx])).long()
    CLIM, ASD = to_t(clim), to_t(anom_sd)
    x = to_t((Wtr[idx] - mu_x) / sd_x)
    y = (to_t(Ytr[idx]) - CLIM[h]) / ASD
    return {'x': x, 'y': y, 'h': h, 'CLIM': CLIM, 'ASD': ASD,
            'stats': dict(mu_x=mu_x, sd_x=sd_x, clim=clim, anom_sd=anom_sd)}

def perdida(nombre, pred, y, ASD):
    if nombre == 'mse':                       # starter: las 4 anomalías normalizadas pesan igual
        return F.mse_loss(pred, y)
    mse = ((pred - y) ** 2).mean(dim=(0, 1, 2)) * (ASD / ESC) ** 2   # por variable, en unidades de la métrica
    if nombre == 'wmse':
        return mse.mean()
    if nombre == 'rmse':                      # 1 - score del lote
        return torch.sqrt(mse + 1e-8).mean()
    raise ValueError(nombre)

def entrenar(D, epocas, nombre_perdida, semilla, ema_decay=0.995, lote=32):
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    rng = np.random.default_rng(semilla)
    model = Net().to(dev)
    ema = {k: v.detach().clone() for k, v in model.state_dict().items()}
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    n = len(D['x']); pasos = epocas * int(np.ceil(n / lote))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=pasos)
    paso = 0
    for ep in range(epocas):
        model.train()
        orden = torch.tensor(rng.permutation(n), device=dev)
        for s in range(0, n, lote):
            b = orden[s:s + lote]
            loss = perdida(nombre_perdida, model(D['x'][b], D['h'][b].float()), D['y'][b], D['ASD'])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            paso += 1
            d = min(ema_decay, paso / (paso + 10))
            with torch.no_grad():
                for k, v in model.state_dict().items():
                    ema[k].lerp_(v, 1 - d)
    modelo_ema = Net().to(dev); modelo_ema.load_state_dict(ema)
    return model, modelo_ema

def predecir_lote(model, D, W, t, bs=128):
    s = D['stats']; model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), bs):
            x = to_t((W[i:i + bs] - s['mu_x']) / s['sd_x'])
            h = to_t(horas(t[i:i + bs])).long()
            out.append((D['CLIM'][h] + model(x, h.float()) * D['ASD']).cpu().numpy())
    return np.concatenate(out)

def recortar_hr(P):
    P = P.copy(); P[..., 0] = np.clip(P[..., 0], 0, 100)
    return P

def medir(P):
    r = validar(P, Ytr, ttr)
    return {'score_oficial': r['score_oficial'], 'score_local': r['score_local'], 'rmse_3h': r['rmse_horas_evaluadas']}

def linea(etiqueta, m):
    v = ' '.join(f'{k} {x:.3f}' for k, x in m['rmse_3h'].items())
    print(f"{etiqueta:34s} oficial {m['score_oficial']:.5f} · local {m['score_local']:.5f} · {v}", flush=True)

In [ ]:
D = datos_de(trn)
PERDIDAS = ['mse', 'wmse', 'rmse']
EPOCAS = [2] if QUICK else [25, 60, 120, 240]
SEMILLAS = [0] if QUICK else [0, 1]
filas, preds = [], {}
for ep in EPOCAS:
    for nombre in PERDIDAS:
        for semilla in SEMILLAS:
            t0 = time.time()
            m, m_ema = entrenar(D, ep, nombre, semilla)
            for pesos, mod in (('final', m), ('ema', m_ema)):
                P = predecir_lote(mod, D, Wtr[val], ttr[val])
                for hr, Q in (('sin_recorte', P), ('hr_0_100', recortar_hr(P))):
                    r = medir(Q)
                    filas.append(dict(perdida=nombre, epocas=ep, semilla=semilla, pesos=pesos, hr=hr, **r))
                linea(f'{nombre} {ep:3d}ep s{semilla} {pesos}', r)
            print(f'  ({time.time() - t0:.0f}s)', flush=True)
            json.dump(filas, open('resultados_a.json', 'w'), indent=1)

tabla = pd.DataFrame(filas)
resumen = tabla.groupby(['perdida', 'epocas', 'pesos', 'hr'])['score_oficial'].agg(['mean', 'min', 'max']).sort_values('mean', ascending=False)
print(resumen.round(5).to_string())

In [ ]:
MEJOR = dict(zip(['perdida', 'epocas', 'pesos', 'hr'], resumen.index[0]))
MEJOR['epocas'] = int(MEJOR['epocas'])
print('Mejor configuración (media de semillas):', MEJOR)

def pred_config(D, semilla, W, t):
    m, m_ema = entrenar(D, MEJOR['epocas'], MEJOR['perdida'], semilla)
    mod = m_ema if MEJOR['pesos'] == 'ema' else m
    P = predecir_lote(mod, D, W, t)
    return mod, (recortar_hr(P) if MEJOR['hr'] == 'hr_0_100' else P)

N_ENS = 2 if QUICK else 5
suma, ensamble = 0, []
for semilla in range(N_ENS):
    _, P = pred_config(D, semilla, Wtr[val], ttr[val])
    suma = suma + P
    solo, junto = medir(P), medir(suma / (semilla + 1))
    ensamble.append(dict(semillas=semilla + 1, solo=solo, ensamble=junto))
    linea(f'semilla {semilla} sola', solo)
    linea(f'ensamble de {semilla + 1}', junto)

# Reproducibilidad: repetir la semilla 0 y comparar
_, P0 = pred_config(D, 0, Wtr[val], ttr[val])
repro = {'primera': ensamble[0]['solo']['score_oficial'], 'repetida': medir(P0)['score_oficial']}
print('Reproducibilidad semilla 0:', repro)
json.dump({'mejor': MEJOR, 'ensamble': ensamble, 'reproducibilidad': repro,
           'versiones': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__,
                         'gpu': torch.cuda.get_device_name(0)}}, open('resultados_b.json', 'w'), indent=1)
del D; torch.cuda.empty_cache()

In [ ]:
# Reentrenamiento con TODOS los datos (incluye la validación): ya no hay score local que mirar.
D = datos_de(np.ones(len(ttr), bool))
os.makedirs('models', exist_ok=True)
suma = 0
for semilla in range(N_ENS):
    mod, P = pred_config(D, semilla, Wte, tte)
    suma = suma + P
    torch.save(mod.state_dict(), f'models/modelo_s{semilla}.pt')
    if semilla == 0:
        P_uno = P
np.savez('models/estadisticas.npz', **D['stats'])
sample = pd.read_csv(find('sample_submission.csv'))
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti = parts[0].map(pos).values; r = parts[1].astype(int).values; c = parts[2].astype(int).values
for archivo, P in (('submission_ensamble.csv', suma / N_ENS), ('submission_semilla0.csv', P_uno)):
    sub = pd.DataFrame({'id': sample['id']})
    for k, v in enumerate(VARS):
        sub[v] = P[ti, r, c, k].round(3)
    assert len(sub) == len(sample) and np.isfinite(sub[list(VARS)].values).all()
    sub.to_csv(archivo, index=False)
    print(archivo, sub.shape)